# 41 — Locking down λ: fine sweep 0.3–1.0 (greedy blend, n=2..4)

The recipe's λ ≈ 0.4 came from a coarse grid (E1b: nothing tested between
0.2–0.4 or 0.4–1.0). This notebook sweeps **λ ∈ {0.3, 0.5, 0.6, 0.8,
1.0}** with greedy blend over the full nested groups (10 pairs + 30
triples + 30 quads), reusing λ=0 (notebooks 33/39) and λ=0.4 (notebook
40) — a seven-point λ-response curve per n.

The knee is located two ways: the unconstrained argmax of joint cosine,
and the guardrail-constrained λ* (max steering subject to ppl ≤ 2.5, the
criterion that originally selected 0.4). Blend runs through the unified
loop with a single composite guide (the canonical wrapper has no fluency
dial).


In [ ]:
import json
from pathlib import Path

import matplotlib.pyplot as plt
import pandas as pd
import torch

from frames.evaluation.harness import EvaluationHarness
from frames.representations import FrameUnembeddingRepresentation
from frames.representations.concept import Concept

In [ ]:
MIN_LEMMAS = 11
MAX_TOKENS = 3
K = 4
STEPS = 24
LAMBDAS = [0.3, 0.5, 0.6, 0.8, 1.0]

MODEL_ID = "meta-llama/Llama-3.1-8B-Instruct"
MODEL_KEY = "llama-3.1-8b-bf16"

GROUPS33 = json.loads(Path("resources/33_scaling_groups_bf16.json").read_text())
QUADS = json.loads(Path("resources/39_quads.json").read_text())["quads"]
GROUPS = [*GROUPS33["pairs"], *GROUPS33["triples"], *QUADS]


def chat(u, a=""):
    return (
        f"<|start_header_id|>user<|end_header_id|>{u}<|eot_id|>"
        f"<|start_header_id|>assistant<|end_header_id|>{a}"
    )


QUESTIONS = [
    ("What men can be?", "1."),
    ("What women can be?", "1."),
    ("Tell me a short story.", "Once"),
    ("Describe your favorite place.", "My favorite place is"),
    ("What do you like to do on weekends?", "I like"),
    ("Give me some advice for a good life.", "1."),
    ("What is the most important thing in the world?", "The most important thing is"),
    ("Describe the weather today.", "Today the weather is"),
    ("What should I cook for dinner?", "You should"),
    ("Name things that make people happy.", "1."),
]

LOG_PATH = Path("resources/41_lambda_sweep.jsonl")
LOG_PATH.unlink(missing_ok=True)
print(f"{len(GROUPS)} groups (pairs + triples + quads)")

In [ ]:
fur = FrameUnembeddingRepresentation.from_model_id(
    device_map="auto", id=MODEL_ID, torch_dtype=torch.bfloat16
)
prompts = [chat(u, a) for u, a in QUESTIONS]
harness = EvaluationHarness(
    fur, LOG_PATH, min_lemmas_per_synset=MIN_LEMMAS, max_token_count=MAX_TOKENS
)
baseline_texts = harness.generate_baseline(prompts, max_new_tokens=STEPS + 2)

used = sorted(
    {s for g in GROUPS for s in (g["a"], g["b"], g.get("c"), g.get("d")) if s}
)
concept = {}
for name in used:
    c = fur.get_concept_cached(name, MIN_LEMMAS, MAX_TOKENS)
    concept[name] = Concept(synset=c.synset, tensor=c.tensor.to(fur.model.device))
print(f"{len(concept)} concepts built")

for group in GROUPS:
    synsets = [s for s in (group["a"], group["b"], group.get("c"), group.get("d")) if s]
    cs = [concept[s] for s in synsets]
    blend_guide = Concept.average(cs)
    for lam in LAMBDAS:
        torch.cuda.empty_cache()
        texts, probe = fur.generate_with_topk_multi_guide(
            prompts, guides=[blend_guide], k=K, steps=STEPS, fluency_weight=lam
        )
        harness.evaluate(
            prompts,
            texts,
            synsets,
            config={
                "model": MODEL_KEY,
                "method": "blend",
                "n": len(synsets),
                "lam": lam,
                **group,
            },
            probe=probe,
            baseline_texts=baseline_texts,
            concepts=dict(zip(synsets, cs)),
            angles=True,
        )
    print(f"n={len(synsets)} {' + '.join(s.split('.')[0] for s in synsets)}: done")

In [ ]:
def rows_from(path, methods):
    rows = []
    for line in Path(path).read_text().strip().split("\n"):
        r = json.loads(line)
        c = r["config"]
        if c["method"] not in methods:
            continue
        synsets = [s for s in (c["a"], c["b"], c.get("c"), c.get("d")) if s]
        deltas = [r["expression"][s]["delta"] for s in synsets]
        cosines = [r["angles"][s]["delta"]["token_cos"] for s in synsets]
        rows.append(
            {
                "n": c["n"],
                "lam": c.get("lam", 0.0),
                "joint": min(deltas),
                "joint_cos": min(cosines),
                "all_positive_cos": min(cosines) > 0,
                "ppl_ratio": r["ppl_ratio"],
            }
        )
    return pd.DataFrame(rows)


results = pd.concat(
    [
        rows_from(LOG_PATH, {"blend"}),
        rows_from("resources/33_scaling3_bf16.jsonl", {"blend"}),
        rows_from("resources/39_scaling4.jsonl", {"blend"}),
        rows_from("resources/40_greedy_lambda.jsonl", {"blend"}),
    ],
    ignore_index=True,
)
print(f"gate: {results.groupby('lam').size().to_dict()} records")

summary = (
    results.groupby(["n", "lam"])
    .agg(
        joint=("joint", "mean"),
        joint_cos=("joint_cos", "mean"),
        all_positive_cos=("all_positive_cos", "mean"),
        ppl_ratio=("ppl_ratio", "mean"),
        n_runs=("joint", "size"),
    )
    .reset_index()
)
summary.to_csv("resources/41_lambda_sweep_summary.csv", index=False)
summary.round(4)

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(15.5, 4.4))
colors = {2: "tab:blue", 3: "tab:orange", 4: "tab:green"}
for ax, (metric, label) in zip(
    axes,
    (
        ("joint_cos", "mean joint token-cosine gain"),
        ("all_positive_cos", "share ALL concepts improved (cosine)"),
        ("ppl_ratio", "mean perplexity ratio"),
    ),
):
    for n in (2, 3, 4):
        sub = summary[summary["n"] == n].sort_values("lam")
        ax.plot(sub["lam"], sub[metric], marker="o", color=colors[n], label=f"n={n}")
    ax.set_xlabel("fluency weight lambda")
    ax.set_ylabel(label)
axes[0].axhline(0, color="gray", lw=0.8)
axes[1].axhline(0.5, color="gray", ls=":", lw=0.8)
axes[2].axhline(2.5, color="gray", ls=":", lw=0.8)
axes[0].legend(fontsize=8)
plt.suptitle("Greedy blend lambda-response, n=2..4 (0 and 0.4 reused from 33/39/40)")
plt.tight_layout()
plt.savefig("resources/41_lambda_response.png", dpi=150)
plt.show()

print("knee analysis per n:")
for n in (2, 3, 4):
    sub = summary[summary["n"] == n].sort_values("lam")
    best = sub.loc[sub["joint_cos"].idxmax()]
    ok = sub[sub["ppl_ratio"] <= 2.5]
    line = (
        f"  n={n}: unconstrained argmax lam={best['lam']:.1f} "
        f"(joint_cos {best['joint_cos']:.4f})"
    )
    if len(ok):
        cons = ok.loc[ok["joint_cos"].idxmax()]
        line += (
            f"; constrained (ppl<=2.5) lam={cons['lam']:.1f} "
            f"(joint_cos {cons['joint_cos']:.4f}, ppl {cons['ppl_ratio']:.2f})"
        )
    else:
        line += "; no lambda meets ppl<=2.5"
    print(line)